# Business Understanding

### Background
Maritime transport is a major component of global trade, connecting economies through shipping routes, ports and logistics networks. Understanding how trade volumes, shipping capacity, connectivity and port activity interact is important for analysing the performance and evolution of the global maritime system.

### Problem Statement
Maritime trade data is distributed across multiple sources covering different aspects of the shipping system. Analysing these indicators separately limits the understanding of their relationships, emerging patterns and changes during major disruptions. An integrated analytical approach can provide broader insights into global maritime trade and its future trends.

### Objectives
**General Objective:**  
To analyse global maritime trade patterns, relationships and emerging trends using data analytics, statistical analysis and machine learning.

**Specific Objectives:**
1. Examine historical trends in global maritime trade and related indicators.
2. Investigate relationships among trade, shipping capacity, connectivity and port activity.
3. Analyse the effects of major disruptions on maritime trade and shipping activity.
4. Assess maritime resilience through disruption impact, recovery and stability.
5. Identify unusual port activity and group ports by operational patterns.
6. Develop exploratory forecasts of annual seaborne trade by cargo type and geographical region, where data permits.

### Research Questions
1. How have global maritime trade patterns evolved over time?
2. What relationships exist among trade, shipping capacity, connectivity and port activity?
3. How have major disruptions affected maritime trade and shipping activity?
4. How does the maritime system respond and recover from disruptions?
5. What similarities and unusual activity patterns can be identified across ports?
6. What future trends in seaborne trade by cargo type and region can be estimated?

### Scope and Expected Outcomes
The project uses eight datasets covering seaborne trade, container throughput, merchant fleets, connectivity, port calls, port performance, daily port activity and chokepoint transits.

Expected outcomes include insights into maritime trade patterns, relationships, disruption impacts, port activity and exploratory trade forecasts. Forecasts will be evaluated against a naïve baseline. Findings describe associations rather than establish causation, and forecasts are subject to data coverage and limitations.


# Data Understanding

In [3]:
# imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from statsmodels.tsa.arima.model import ARIMA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error,mean_absolute_percentage_error,silhouette_score,davies_bouldin_score,calinski_harabasz_score)
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


In [4]:
# load daily ports
portwatch = pd.read_csv("Datasets/raw/daily_ports_all.csv")
print(portwatch.shape)
portwatch.head()

(5952054, 30)


,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
0,2021-03-11,2021,3,11,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2020-02-05,2020,2,5,port1325,Tsuruga,Japan,JPN,0,0,...,160,160,0,0,0,543,0,543,543,2
2,2021-03-12,2021,3,12,port1325,Tsuruga,Japan,JPN,0,1,...,242,242,0,1754,0,0,0,1754,1754,3
3,2021-03-13,2021,3,13,port1325,Tsuruga,Japan,JPN,0,0,...,0,0,0,0,0,0,0,0,0,4
4,2021-03-14,2021,3,14,port1325,Tsuruga,Japan,JPN,0,1,...,320,320,0,2670,0,1087,0,3758,3758,5


In [5]:
portwatch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952054 entries, 0 to 5952053
Data columns (total 30 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   date                     object
 1   year                     int64 
 2   month                    int64 
 3   day                      int64 
 4   portid                   object
 5   portname                 object
 6   country                  object
 7   ISO3                     object
 8   portcalls_container      int64 
 9   portcalls_dry_bulk       int64 
 10  portcalls_general_cargo  int64 
 11  portcalls_roro           int64 
 12  portcalls_tanker         int64 
 13  portcalls_cargo          int64 
 14  portcalls                int64 
 15  import_container         int64 
 16  import_dry_bulk          int64 
 17  import_general_cargo     int64 
 18  import_roro              int64 
 19  import_tanker            int64 
 20  import_cargo             int64 
 21  import                   int64 

In [6]:
portwatch.describe()

,year,month,day,portcalls_container,portcalls_dry_bulk,portcalls_general_cargo,portcalls_roro,portcalls_tanker,portcalls_cargo,portcalls,...,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export,ObjectId
count,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,...,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06,5.952054e+06
mean,2.022441e+03,6.437370e+00,1.568963e+01,6.170749e-01,3.757427e-01,5.177514e-01,1.377770e-01,7.559642e-01,1.648346e+00,2.404310e+00,...,1.121031e+04,1.593580e+04,4.275932e+03,6.295131e+03,4.177303e+02,6.763786e+01,4.420149e+03,1.105650e+04,1.547669e+04,2.976028e+06
std,2.264716e+00,3.412146e+00,8.803135e+00,2.478460e+00,1.379177e+00,1.478039e+00,5.896698e-01,2.852803e+00,4.420998e+00,6.530501e+00,...,5.532706e+04,6.991363e+04,3.671674e+04,4.729657e+04,1.743020e+03,5.592610e+02,2.778433e+04,6.194919e+04,7.134544e+04,1.718210e+06
min,2.019000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,2.020000e+03,3.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.488014e+06
50%,2.022000e+03,6.000000e+00,1.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.976028e+06
75%,2.024000e+03,9.000000e+00,2.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,...,1.116000e+03,3.014000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.180000e+02,2.207000e+03,4.464041e+06
max,2.026000e+03,1.200000e+01,3.100000e+01,7.600000e+01,8.300000e+01,4.300000e+01,1.900000e+01,1.170000e+02,1.760000e+02,2.180000e+02,...,3.406220e+06,3.706936e+06,2.004273e+06,2.641160e+06,8.454700e+04,3.543000e+04,5.437958e+06,2.657613e+06,5.877828e+06,5.952054e+06


In [7]:
## check portwatch mising values
portwatch.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [8]:
portwatch.duplicated().sum()

np.int64(0)

In [9]:
# load us seaborne
seaborne = pd.read_csv('Datasets/raw/US.SeaborneTrade_20261110_095251.csv')
print(seaborne.shape)
seaborne.head()

(333, 77)


,Economy_Label,CargoType_Label,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2000_Metric_tons_MissingValue,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2001_Metric_tons_MissingValue,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,...,2021_Metric_tons_MissingValue,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2022_Metric_tons_MissingValue,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2023_Metric_tons_MissingValue,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote,2024_Metric_tons_MissingValue
0,World,Crude oil loaded,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN
1,World,Other tanker trade loaded,1171172.0,NaN,NaN,1162476.0,NaN,NaN,1196826.0,NaN,...,NaN,2290940.0,NaN,NaN,2288012.0,NaN,NaN,2284711.0,NaN,NaN
2,World,Dry cargo loaded,3361790.0,NaN,NaN,3416510.0,NaN,NaN,3591200.0,NaN,...,NaN,7433784.0,NaN,NaN,7239276.0,NaN,NaN,7726790.0,NaN,NaN
3,World,Total goods loaded,6345730.0,NaN,NaN,6398217.0,NaN,NaN,6576741.0,NaN,...,NaN,11754284.0,NaN,NaN,11598777.0,NaN,NaN,12054069.0,NaN,NaN
4,World,Crude oil discharged,1812768.0,NaN,NaN,1819232.0,NaN,NaN,1788715.0,NaN,...,NaN,2029559.0,NaN,NaN,2071488.0,NaN,NaN,2042568.0,NaN,NaN


In [10]:
seaborne.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 77 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Economy_Label                        333 non-null    object 
 1   CargoType_Label                      333 non-null    object 
 2   2000_Metric_tons_in_thousands_Value  323 non-null    float64
 3   2000_Metric_tons_Footnote            0 non-null      float64
 4   2000_Metric_tons_MissingValue        9 non-null      object 
 5   2001_Metric_tons_in_thousands_Value  322 non-null    float64
 6   2001_Metric_tons_Footnote            0 non-null      float64
 7   2001_Metric_tons_MissingValue        9 non-null      object 
 8   2002_Metric_tons_in_thousands_Value  324 non-null    float64
 9   2002_Metric_tons_Footnote            0 non-null      float64
 10  2002_Metric_tons_MissingValue        9 non-null      object 
 11  2003_Metric_tons_in_thousands_Va

In [11]:
seaborne.describe()

,2000_Metric_tons_in_thousands_Value,2000_Metric_tons_Footnote,2001_Metric_tons_in_thousands_Value,2001_Metric_tons_Footnote,2002_Metric_tons_in_thousands_Value,2002_Metric_tons_Footnote,2003_Metric_tons_in_thousands_Value,2003_Metric_tons_Footnote,2004_Metric_tons_in_thousands_Value,2004_Metric_tons_Footnote,...,2020_Metric_tons_in_thousands_Value,2020_Metric_tons_Footnote,2021_Metric_tons_in_thousands_Value,2021_Metric_tons_Footnote,2022_Metric_tons_in_thousands_Value,2022_Metric_tons_Footnote,2023_Metric_tons_in_thousands_Value,2023_Metric_tons_Footnote,2024_Metric_tons_in_thousands_Value,2024_Metric_tons_Footnote
count,3.230000e+02,0.0,3.220000e+02,0.0,3.240000e+02,0.0,3.220000e+02,0.0,3.230000e+02,0.0,...,3.230000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0,3.240000e+02,0.0
mean,9.871586e+05,NaN,9.974137e+05,NaN,1.019340e+06,NaN,1.080934e+06,NaN,1.166356e+06,NaN,...,1.830849e+06,NaN,1.900331e+06,NaN,1.906340e+06,NaN,1.883756e+06,NaN,1.966969e+06,NaN
std,1.533092e+06,NaN,1.545379e+06,NaN,1.585122e+06,NaN,1.669515e+06,NaN,1.801689e+06,NaN,...,2.829516e+06,NaN,2.939173e+06,NaN,2.938670e+06,NaN,2.901268e+06,NaN,3.051610e+06,NaN
min,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,...,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN
25%,6.204950e+04,NaN,6.161550e+04,NaN,5.827925e+04,NaN,6.325275e+04,NaN,6.363000e+04,NaN,...,1.232500e+05,NaN,1.228542e+05,NaN,1.261062e+05,NaN,1.285468e+05,NaN,1.338748e+05,NaN
50%,4.190540e+05,NaN,4.285210e+05,NaN,4.487830e+05,NaN,4.721545e+05,NaN,5.050890e+05,NaN,...,7.404920e+05,NaN,7.616535e+05,NaN,7.510270e+05,NaN,7.722395e+05,NaN,7.890845e+05,NaN
75%,1.215596e+06,NaN,1.215856e+06,NaN,1.232880e+06,NaN,1.314100e+06,NaN,1.405542e+06,NaN,...,2.205966e+06,NaN,2.326580e+06,NaN,2.314270e+06,NaN,2.308013e+06,NaN,2.404866e+06,NaN
max,1.269146e+07,NaN,1.279643e+07,NaN,1.315348e+07,NaN,1.386194e+07,NaN,1.498393e+07,NaN,...,2.263529e+07,NaN,2.354976e+07,NaN,2.350857e+07,NaN,2.319755e+07,NaN,2.410814e+07,NaN


In [12]:
# missing values in seaborne
seaborne.isnull().sum()

Economy_Label                            0
CargoType_Label                          0
2000_Metric_tons_in_thousands_Value     10
2000_Metric_tons_Footnote              333
2000_Metric_tons_MissingValue          324
                                      ... 
2023_Metric_tons_Footnote              333
2023_Metric_tons_MissingValue          324
2024_Metric_tons_in_thousands_Value      9
2024_Metric_tons_Footnote              333
2024_Metric_tons_MissingValue          324
Length: 77, dtype: int64

In [13]:
seaborne.duplicated().sum()

np.int64(0)

In [14]:
# load merchant fleet
fleet = pd.read_csv('Datasets/raw/US_MerchantFleet.csv')
print(fleet.shape)
fleet.head()

(63896, 23)


,Year,Economy,Economy Label,ShipType,ShipType Label,Dead weight tons in thousands,Dead weight tons in thousands Footnote,Dead weight tons in thousands Missing value,Percentage of total world,Percentage of total world Footnote,...,Percentage of total fleet Missing value,Number of ships,Number of ships Footnote,Number of ships Missing value,Gross Tonnage in thousands,Gross Tonnage in thousands Footnote,Gross Tonnage in thousands Missing value,Average age of vessels (years),Average age of vessels (years) Footnote,Average age of vessels (years) Missing value
0,1980,0000,World,0,Total fleet,672192.465,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
1,1980,0000,World,1,Oil tankers,337945.534,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
2,1980,0000,World,2,Bulk carriers,181880.282,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
3,1980,0000,World,3,General cargo,112840.897,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
4,1980,0000,World,4,Container ships,10290.114,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected


In [15]:
fleet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63896 entries, 0 to 63895
Data columns (total 23 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Year                                          63896 non-null  int64  
 1   Economy                                       63896 non-null  object 
 2   Economy Label                                 63896 non-null  object 
 3   ShipType                                      63896 non-null  int64  
 4   ShipType Label                                63896 non-null  object 
 5   Dead weight tons in thousands                 58814 non-null  float64
 6   Dead weight tons in thousands Footnote        24 non-null     object 
 7   Dead weight tons in thousands Missing value   5082 non-null   object 
 8   Percentage of total world                     58814 non-null  float64
 9   Percentage of total world Footnote            24 non-null    

In [16]:
fleet.describe()

,Year,ShipType,Dead weight tons in thousands,Percentage of total world,Percentage of total fleet,Number of ships,Gross Tonnage in thousands,Average age of vessels (years)
count,63896.000000,63896.000000,5.881400e+04,58814.000000,58814.000000,24600.000000,2.460000e+04,20023.000000
mean,2004.610852,2.457759,2.174818e+04,5.397872,37.797279,1506.377154,2.017613e+04,24.460755
std,13.783127,1.748450,8.731343e+04,12.848273,37.409220,5927.114144,7.814213e+04,11.648492
min,1980.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1993.000000,1.000000,1.753525e+01,0.008471,5.930415,2.000000,2.652000e+00,16.080000
50%,2006.000000,2.000000,4.118255e+02,0.188330,22.924635,35.000000,1.707100e+02,23.290000
75%,2017.000000,4.000000,6.420298e+03,3.591282,66.899630,543.250000,4.086645e+03,30.906500
max,2026.000000,5.000000,2.528788e+06,100.000000,100.000000,115649.000000,1.737514e+06,109.920000


In [17]:
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [18]:
fleet.duplicated().sum()

np.int64(0)

In [19]:
# load lsci
lsci = pd.read_csv('Datasets/raw/US.LSCI_20261121_091634.csv')
print(lsci.shape)
lsci.head()

(15013, 5)


,Economy_Label,Quarter_Label,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote,Index_average_Q1_2023__100_MissingValue
0,Albania,Q4 2006,5.01,NaN,NaN
1,Albania,Q1 2007,5.48,NaN,NaN
2,Albania,Q2 2007,5.48,NaN,NaN
3,Albania,Q3 2007,5.04,NaN,NaN
4,Albania,Q4 2007,5.19,NaN,NaN


In [20]:
lsci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15013 entries, 0 to 15012
Data columns (total 5 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Economy_Label                            15013 non-null  object 
 1   Quarter_Label                            15013 non-null  object 
 2   Index_average_Q1_2023__100_Value         14724 non-null  float64
 3   Index_average_Q1_2023__100_Footnote      0 non-null      float64
 4   Index_average_Q1_2023__100_MissingValue  289 non-null    object 
dtypes: float64(2), object(3)
memory usage: 586.6+ KB


In [21]:
lsci.describe()

,Index_average_Q1_2023__100_Value,Index_average_Q1_2023__100_Footnote
count,14724.000000,0.0
mean,96.932643,NaN
std,133.808579,NaN
min,2.500000,NaN
25%,22.792500,NaN
50%,47.060000,NaN
75%,106.880000,NaN
max,1341.970000,NaN


In [22]:
lsci.isnull().sum()

Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
dtype: int64

In [23]:
lsci.duplicated().sum()

np.int64(0)

In [24]:
# load container throughput
teu = pd.read_csv('Datasets/raw/US_ContPortThroughput.csv')
print(teu.shape)
teu.head()

(3030, 6)


,Year,Economy,Economy Label,TEU (Twenty foot Equivalent Unit),TEU (Twenty foot Equivalent Unit) Footnote,TEU (Twenty foot Equivalent Unit) Missing value
0,2010,0,World,545520944.0,NaN,NaN
1,2010,5100,Africa,23416816.0,NaN,NaN
2,2010,5110,Northern Africa,12024778.0,NaN,NaN
3,2010,5120,Sub-Saharan Africa,11392038.0,NaN,NaN
4,2010,5121,Eastern Africa,2297961.0,NaN,NaN


In [25]:
teu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Year                                             3030 non-null   int64  
 1   Economy                                          3030 non-null   int64  
 2   Economy Label                                    3030 non-null   object 
 3   TEU (Twenty foot Equivalent Unit)                2900 non-null   float64
 4   TEU (Twenty foot Equivalent Unit) Footnote       19 non-null     object 
 5   TEU (Twenty foot Equivalent Unit) Missing value  130 non-null    object 
dtypes: float64(1), int64(2), object(3)
memory usage: 142.2+ KB


In [26]:
teu.describe()

,Year,Economy,TEU (Twenty foot Equivalent Unit)
count,3030.000000,3030.000000,2.900000e+03
mean,2017.000000,1431.004950,4.752831e+07
std,4.321207,1819.965223,1.193780e+08
min,2010.000000,0.000000,9.500000e+03
25%,2013.000000,308.000000,4.678590e+05
50%,2017.000000,612.000000,3.420179e+06
75%,2021.000000,1530.000000,2.146032e+07
max,2024.000000,5706.000000,9.203008e+08


In [27]:
teu.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900
dtype: int64

In [28]:
teu.duplicated().sum()

np.int64(0)

In [29]:
# load chokepoints 
choke = pd.read_csv('Datasets/raw/Daily_Chokepoints_Data.csv')
print(choke.shape)
choke.head()

(80528, 21)


,date,year,month,day,portid,portname,n_container,n_dry_bulk,n_general_cargo,n_roro,...,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
0,2019/01/01,2019,1,1,chokepoint1,Suez Canal,15,15,9,4,...,43,61,1018466,618654,112660,21900,1279775,1771682,3051458,1
1,2021/03/11,2021,3,11,chokepoint1,Suez Canal,16,12,3,2,...,33,56,964354,351621,20733,10326,1021116,1347035,2368151,2
2,2019/01/02,2019,1,2,chokepoint1,Suez Canal,24,4,6,6,...,40,50,1534924,227315,8536,37934,518306,1808711,2327017,3
3,2019/01/03,2019,1,3,chokepoint1,Suez Canal,13,14,9,2,...,38,62,555097,977689,39398,10829,868092,1583014,2451107,4
4,2021/03/12,2021,3,12,chokepoint1,Suez Canal,22,15,3,2,...,42,54,1372498,752113,2341,11084,670038,2138038,2808077,5


In [30]:
choke.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80528 entries, 0 to 80527
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   date                    80528 non-null  object
 1   year                    80528 non-null  int64 
 2   month                   80528 non-null  int64 
 3   day                     80528 non-null  int64 
 4   portid                  80528 non-null  object
 5   portname                80528 non-null  object
 6   n_container             80528 non-null  int64 
 7   n_dry_bulk              80528 non-null  int64 
 8   n_general_cargo         80528 non-null  int64 
 9   n_roro                  80528 non-null  int64 
 10  n_tanker                80528 non-null  int64 
 11  n_cargo                 80528 non-null  int64 
 12  n_total                 80528 non-null  int64 
 13  capacity_container      80528 non-null  int64 
 14  capacity_dry_bulk       80528 non-null  int64 
 15  ca

In [31]:
choke.describe()

,year,month,day,n_container,n_dry_bulk,n_general_cargo,n_roro,n_tanker,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity,ObjectId
count,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,80528.000000,8.052800e+04,8.052800e+04,80528.000000,80528.000000,8.052800e+04,8.052800e+04,8.052800e+04,80528.000000
mean,2022.443672,6.440542,15.688804,14.387666,21.664167,11.469278,2.315766,20.811072,49.836877,70.647949,3.667380e+05,1.097857e+06,38229.182036,5440.079053,5.997501e+05,1.508265e+06,2.108016e+06,40264.500000
std,2.265885,3.413100,8.800194,20.696034,23.221566,15.890087,3.221351,22.203804,51.798241,71.148095,5.214239e+05,1.282904e+06,48834.982214,8670.923652,8.209306e+05,1.581379e+06,2.105823e+06,23246.575576
min,2019.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000
25%,2020.000000,3.000000,8.000000,1.000000,4.000000,1.000000,0.000000,4.000000,12.000000,16.000000,1.158750e+04,1.355515e+05,3470.000000,0.000000,3.459850e+04,2.281238e+05,2.912115e+05,20132.750000
50%,2022.000000,6.000000,16.000000,6.000000,13.000000,4.000000,1.000000,12.000000,32.000000,46.000000,1.007985e+05,5.545225e+05,17303.500000,0.000000,2.563505e+05,8.743895e+05,1.406020e+06,40264.500000
75%,2024.000000,9.000000,23.000000,17.000000,32.000000,17.000000,3.000000,30.000000,66.000000,95.000000,5.417912e+05,1.732328e+06,55880.250000,7942.500000,8.677295e+05,2.327564e+06,3.258355e+06,60396.250000
max,2026.000000,12.000000,31.000000,172.000000,245.000000,149.000000,23.000000,148.000000,352.000000,472.000000,4.016066e+06,1.317163e+07,388902.000000,83176.000000,6.596353e+06,1.434564e+07,1.632013e+07,80528.000000


In [32]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [33]:
choke.duplicated().sum()

np.int64(0)

In [34]:
# load port calls
calls = pd.read_csv('Datasets/raw/US_PortCallsArrivals_S.csv')
print(calls.shape)
calls.head()

(9300, 9)


,Period,Period Label,Economy,Economy Label,CommercialMarket,CommercialMarket Label,Number of port calls,Number of port calls Footnote,Number of port calls Missing value
0,2018S01,S1 2018,1,Individual economies,0,All ships,NaN,NaN,Not applicable
1,2018S01,S1 2018,1,Individual economies,2,Liquid bulk carriers,NaN,NaN,Not applicable
2,2018S01,S1 2018,1,Individual economies,7,Liquefied petroleum gas carriers,NaN,NaN,Not applicable
3,2018S01,S1 2018,1,Individual economies,8,Liquefied natural gas carriers,NaN,NaN,Not applicable
4,2018S01,S1 2018,1,Individual economies,5,Dry bulk carriers,NaN,NaN,Not applicable


In [35]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9300 entries, 0 to 9299
Data columns (total 9 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Period                              9300 non-null   object 
 1   Period Label                        9300 non-null   object 
 2   Economy                             9300 non-null   int64  
 3   Economy Label                       9300 non-null   object 
 4   CommercialMarket                    9300 non-null   int64  
 5   CommercialMarket Label              9300 non-null   object 
 6   Number of port calls                8810 non-null   float64
 7   Number of port calls Footnote       0 non-null      float64
 8   Number of port calls Missing value  490 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 654.0+ KB


In [36]:
calls.describe()

,Economy,CommercialMarket,Number of port calls,Number of port calls Footnote
count,9300.000000,9300.000000,8.810000e+03,0.0
mean,2821.193011,3.962903,6.490234e+04,NaN
std,2112.189066,2.568352,1.991938e+05,NaN
min,0.000000,0.000000,1.100000e+01,NaN
25%,842.000000,2.000000,1.380500e+03,NaN
50%,1640.000000,4.000000,7.185000e+03,NaN
75%,5221.000000,6.000000,4.047450e+04,NaN
max,5900.000000,8.000000,2.461711e+06,NaN


In [37]:
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [38]:
choke.duplicated().sum()

np.int64(0)

In [39]:
# load cppi 
cppi = pd.read_excel('Datasets/raw/CPPI 2020-2025.xlsx')
print(cppi.shape)
cppi.head()

(426, 20)


,Port,UN/LOCODE,Territory,World Bank Region,Region average CPPI,World Bank Income Group,Income Group average CPPI,Maritime services region,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
0,Aarhus,DKAAR,Denmark,ECA,-8.086595,HI,1.362073,North Europe,-7.718713,73.0,38.0,22.0,60.0,99.0,86.634702,33.0,0.856896,47.61,125.659404,292.0
1,Abidjan,CIABJ,Côte d’Ivoire,SSA,-71.972350,LMI,-8.954771,West Africa,-66.152693,19.0,-153.0,-144.0,-38.0,-51.0,-127.472673,381.0,0.595764,-70.25,-184.695347,407.0
2,Acajutla,SVAQJ,El Salvador,LAC,-14.986492,UMI,1.615451,Caribbean & Central America,-19.239499,0.0,-7.0,-30.0,-89.0,-98.0,-207.620361,393.0,0.479969,-143.38,-271.860723,71.0
3,Adelaide,AUADL,Australia,EAP,30.729639,HI,1.362073,Australasia & Oceania,-11.011090,-20.0,-2.0,-24.0,-39.0,-8.0,-18.214819,297.0,0.833064,-11.43,-24.999638,213.0
4,Aden,YEADE,Yemen,MENAAP,18.958816,LI,-23.905865,Red Sea,63.164190,NaN,-21.0,-13.0,4.0,-24.0,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
cppi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426 entries, 0 to 425
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Port                            424 non-null    object 
 1   UN/LOCODE                       419 non-null    object 
 2   Territory                       419 non-null    object 
 3   World Bank Region               419 non-null    object 
 4   Region average CPPI             419 non-null    float64
 5   World Bank Income Group         419 non-null    object 
 6   Income Group average CPPI       419 non-null    float64
 7   Maritime services region        419 non-null    object 
 8   Region's average cppi           419 non-null    float64
 9   CPPI 2020                       326 non-null    float64
 10  CPPI 2021                       352 non-null    float64
 11  CPPI 2022                       340 non-null    float64
 12  CPPI 2023                       391 

In [41]:
cppi.describe()

,Region average CPPI,Income Group average CPPI,Region's average cppi,CPPI 2020,CPPI 2021,CPPI 2022,CPPI 2023,CPPI 2024,CPPI 2025,Rank 2025,Berth hours in % of port hours,Statistical Index,Administrative Index,Number of calls in sample
count,419.000000,419.000000,419.000000,326.000000,352.000000,340.000000,391.000000,403.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,-1.341200,-1.359779,-0.973744,37.710675,5.373509,-2.009647,6.866905,0.012407,-1.288518,200.500000,0.756255,-1.871100,-0.705937,449.247500
std,29.907600,6.021963,36.738435,48.038120,80.082824,78.582603,68.823008,71.335361,64.520553,115.614301,0.136932,38.801558,90.544433,782.571439
min,-71.972350,-25.469787,-98.080938,-108.000000,-665.000000,-675.000000,-519.000000,-721.000000,-302.583415,1.000000,0.369475,-176.720000,-428.446829,24.000000
25%,-14.986492,1.362073,-19.239499,6.000000,-7.000000,-16.000000,-12.500000,-16.500000,-19.752567,100.750000,0.673786,-12.080000,-25.612862,81.750000
50%,-8.086595,1.362073,-7.718713,24.000000,12.000000,7.860000,6.000000,4.000000,1.371539,200.500000,0.785014,1.000000,2.509513,194.000000
75%,30.729639,1.615451,17.910438,68.000000,39.000000,34.000000,39.000000,27.000000,24.013080,300.250000,0.860420,13.542500,33.262400,457.000000
max,34.424249,1.615451,103.168045,170.000000,152.000000,139.000000,152.000000,146.000000,144.638476,400.000000,0.983194,83.550000,209.356951,7385.000000


In [42]:
cppi.isnull().sum()

Port                                2
UN/LOCODE                           7
Territory                           7
World Bank Region                   7
Region average CPPI                 7
World Bank Income Group             7
Income Group average CPPI           7
Maritime services region            7
Region's average cppi               7
CPPI 2020                         100
CPPI 2021                          74
CPPI 2022                          86
CPPI 2023                          35
CPPI 2024                          23
CPPI 2025                          26
Rank 2025                          26
Berth hours in % of port hours     26
Statistical Index                  26
Administrative Index               26
Number of calls in sample          26
dtype: int64

In [43]:
cppi.duplicated().sum()

np.int64(1)

# Data Preparation.

In [44]:
# portwatch 
portwatch_raw= portwatch.copy()

In [45]:
portwatch_raw.isnull().sum()

date                       0
year                       0
month                      0
day                        0
portid                     0
portname                   0
country                    0
ISO3                       0
portcalls_container        0
portcalls_dry_bulk         0
portcalls_general_cargo    0
portcalls_roro             0
portcalls_tanker           0
portcalls_cargo            0
portcalls                  0
import_container           0
import_dry_bulk            0
import_general_cargo       0
import_roro                0
import_tanker              0
import_cargo               0
import                     0
export_container           0
export_dry_bulk            0
export_general_cargo       0
export_roro                0
export_tanker              0
export_cargo               0
export                     0
ObjectId                   0
dtype: int64

In [186]:
# Convert date
portwatch["date"] = pd.to_datetime(portwatch["date"])

# Identify partial year
latest_year = portwatch["date"].dt.year.max()

portwatch["partial_year"] = (portwatch["date"].dt.year == latest_year)

# Remove redundant columns
portwatch.drop(columns=["year", "month", "day", "ObjectId"],inplace=True)

print("Cleaned shape:", portwatch.shape)
print("Columns:", portwatch.columns.tolist())

Cleaned shape: (5952054, 27)
Columns: ['date', 'portid', 'portname', 'country', 'ISO3', 'portcalls_container', 'portcalls_dry_bulk', 'portcalls_general_cargo', 'portcalls_roro', 'portcalls_tanker', 'portcalls_cargo', 'portcalls', 'import_container', 'import_dry_bulk', 'import_general_cargo', 'import_roro', 'import_tanker', 'import_cargo', 'import', 'export_container', 'export_dry_bulk', 'export_general_cargo', 'export_roro', 'export_tanker', 'export_cargo', 'export', 'partial_year']


In [46]:
# Seaborne
seaborne.columns

Index(['Economy_Label', 'CargoType_Label',
       '2000_Metric_tons_in_thousands_Value', '2000_Metric_tons_Footnote',
       '2000_Metric_tons_MissingValue', '2001_Metric_tons_in_thousands_Value',
       '2001_Metric_tons_Footnote', '2001_Metric_tons_MissingValue',
       '2002_Metric_tons_in_thousands_Value', '2002_Metric_tons_Footnote',
       '2002_Metric_tons_MissingValue', '2003_Metric_tons_in_thousands_Value',
       '2003_Metric_tons_Footnote', '2003_Metric_tons_MissingValue',
       '2004_Metric_tons_in_thousands_Value', '2004_Metric_tons_Footnote',
       '2004_Metric_tons_MissingValue', '2005_Metric_tons_in_thousands_Value',
       '2005_Metric_tons_Footnote', '2005_Metric_tons_MissingValue',
       '2006_Metric_tons_in_thousands_Value', '2006_Metric_tons_Footnote',
       '2006_Metric_tons_MissingValue', '2007_Metric_tons_in_thousands_Value',
       '2007_Metric_tons_Footnote', '2007_Metric_tons_MissingValue',
       '2008_Metric_tons_in_thousands_Value', '2008_Metric_tons_F

In [47]:
# Create a copy of the original dataset
seaborne_clean = seaborne.copy()

# Identify the annual trade-value columns
value_cols = [col for col in seaborne_clean.columns
    if col.endswith("_Metric_tons_in_thousands_Value")
]

# Reshape from wide format to long format
seaborne_clean = seaborne_clean.melt(id_vars=["Economy_Label", "CargoType_Label"],value_vars=value_cols,var_name="Year",value_name="Trade_thousand_tonnes")

# Extract year from column names
seaborne_clean["Year"] = (seaborne_clean["Year"].str.extract(r"(\d{4})").astype(int))

# Rename columns
seaborne_clean = seaborne_clean.rename(columns={"Economy_Label": "Economy","CargoType_Label": "Cargo_type"})

# Arrange columns
seaborne_clean = seaborne_clean[["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]]

# Convert trade values to numeric
seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"],errors="coerce")

print("Standardized columns:")
print(seaborne_clean.columns.tolist())
seaborne_clean.head()

Standardized columns:
['Year', 'Economy', 'Cargo_type', 'Trade_thousand_tonnes']


,Year,Economy,Cargo_type,Trade_thousand_tonnes
0,2000,World,Crude oil loaded,1812768.0
1,2000,World,Other tanker trade loaded,1171172.0
2,2000,World,Dry cargo loaded,3361790.0
3,2000,World,Total goods loaded,6345730.0
4,2000,World,Crude oil discharged,1812768.0


In [48]:
# Clean categorical values
# Preserve official names and meaningful differences between economy groups.
categorical_columns = ["Economy", "Cargo_type"]

for column in categorical_columns:
    seaborne_clean[column] = (seaborne_clean[column].astype("string").str.normalize("NFC").str.replace("\u00a0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA))

In [49]:
# Convert numeric columns and handle non-finite values
seaborne_clean["Year"] = pd.to_numeric(seaborne_clean["Year"], errors="coerce")

seaborne_clean["Trade_thousand_tonnes"] = pd.to_numeric(seaborne_clean["Trade_thousand_tonnes"], errors="coerce")

seaborne_clean = seaborne_clean.replace([np.inf, -np.inf], np.nan)

print("Missing values before removal:")
seaborne_clean.isna().sum()

Missing values before removal:


Year                       0
Economy                    0
Cargo_type                 0
Trade_thousand_tonnes    241
dtype: int64

In [50]:
# Remove rows containing missing values
# Do not replace missing trade values with zero.

rows_before = len(seaborne_clean)

seaborne_clean = seaborne_clean.dropna(subset=["Year", "Economy", "Cargo_type", "Trade_thousand_tonnes"]).copy()

print("Rows removed:", rows_before - len(seaborne_clean))

Rows removed: 241


In [51]:
# Validate years and trade values
invalid_rows = (
    ~seaborne_clean["Year"].between(2000, 2024)
    | seaborne_clean["Year"].mod(1).ne(0)
    | seaborne_clean["Trade_thousand_tonnes"].lt(0)
)

print("Invalid rows:", invalid_rows.sum())

seaborne_clean = seaborne_clean.loc[~invalid_rows].copy()

# Zero trade values are valid and are retained.
seaborne_clean["Year"] = seaborne_clean["Year"].astype("int16")

seaborne_clean["Trade_thousand_tonnes"] = (seaborne_clean["Trade_thousand_tonnes"].astype("float64"))

Invalid rows: 0


In [52]:
# Remove duplicate records
key_columns = ["Year", "Economy", "Cargo_type"]

rows_before = len(seaborne_clean)

# Check whether duplicate keys have conflicting trade values
conflicts = (seaborne_clean.groupby(key_columns)["Trade_thousand_tonnes"].nunique())

if conflicts.gt(1).any():
    print("Warning: Duplicate keys have different trade values.")
    print(seaborne_clean[seaborne_clean.duplicated(key_columns, keep=False)].sort_values(key_columns))
else:
    seaborne_clean = seaborne_clean.drop_duplicates(subset=key_columns,keep="first").copy()

    print("Duplicate rows removed:", rows_before - len(seaborne_clean))

Duplicate rows removed: 0


In [53]:
# Detect potential outliers within each economy and cargo type
# A global Z-score would mix economies of different sizes.
# Use group-specific Z-scores for review only.

grouped_values = seaborne_clean.groupby(["Economy", "Cargo_type"])["Trade_thousand_tonnes"]

group_mean = grouped_values.transform("mean")
group_std = grouped_values.transform("std").replace(0, np.nan)

z_scores = (seaborne_clean["Trade_thousand_tonnes"] - group_mean) / group_std

potential_outliers = (seaborne_clean.loc[z_scores.abs().gt(3)].assign(z_score=z_scores.loc[z_scores.abs().gt(3)]).sort_values("z_score",key=lambda values: values.abs(),ascending=False))

print("Potential outliers requiring review:", len(potential_outliers))
print(potential_outliers.to_string(index=False))

Potential outliers requiring review: 6
 Year                                           Economy                Cargo_type  Trade_thousand_tonnes  z_score
 2024 SIDS (Small island developing States) (UN-OHRLLS)          Crude oil loaded                33922.0 3.721186
 2008                           LDCs: Islands and Haiti      Crude oil discharged                   75.0 3.701860
 2024                                   SIDS: Caribbean          Crude oil loaded                32264.0 3.701154
 2023                                        LDCs: Asia      Crude oil discharged                 1948.0 3.130640
 2008                                            Africa Other tanker trade loaded               142454.0 3.113817
 2008                      Developing economies: Africa Other tanker trade loaded               142412.0 3.113583


In [54]:
# Finalize categorical data types and sort
for column in categorical_columns:
    seaborne_clean[column] = seaborne_clean[column].astype("category")

seaborne_clean = (seaborne_clean.sort_values(["Economy", "Cargo_type", "Year"]).reset_index(drop=True))

In [55]:
# Validate the cleaned dataset
key_columns = ["Year", "Economy", "Cargo_type"]

assert seaborne_clean.notna().all().all()
assert not seaborne_clean.duplicated(subset=key_columns).any()
assert seaborne_clean["Year"].between(2000, 2024).all()
assert seaborne_clean["Trade_thousand_tonnes"].ge(0).all()
assert np.isfinite(seaborne_clean["Trade_thousand_tonnes"]).all()

print("Final shape:", seaborne_clean.shape)

print("\nData types:")
print(seaborne_clean.dtypes)

print("\nMissing values:")
print(seaborne_clean.isna().sum())

Final shape: (8084, 4)

Data types:
Year                        int16
Economy                  category
Cargo_type               category
Trade_thousand_tonnes     float64
dtype: object

Missing values:
Year                     0
Economy                  0
Cargo_type               0
Trade_thousand_tonnes    0
dtype: int64


In [56]:
# Save cleaned dataset in processed folder
seaborne_clean.to_csv("Datasets/processed/SeaborneTrade_Cleaned.csv",index=False)

print("Saved: Datasets/processed/SeaborneTrade_Cleaned.csv")

Saved: Datasets/processed/SeaborneTrade_Cleaned.csv


In [57]:
# merchant fleet
fleet.isnull().sum()

Year                                                0
Economy                                             0
Economy Label                                       0
ShipType                                            0
ShipType Label                                      0
Dead weight tons in thousands                    5082
Dead weight tons in thousands Footnote          63872
Dead weight tons in thousands Missing value     58814
Percentage of total world                        5082
Percentage of total world Footnote              63872
Percentage of total world Missing value         58814
Percentage of total fleet                        5082
Percentage of total fleet Footnote              63872
Percentage of total fleet Missing value         58814
Number of ships                                 39296
Number of ships Footnote                        63884
Number of ships Missing value                   24600
Gross Tonnage in thousands                      39296
Gross Tonnage in thousands F

In [58]:
print(fleet.shape)                      
print(fleet["Economy"].head(3).tolist()) 
print(fleet["Economy"].dtype)

(63896, 23)
['0000', '0000', '0000']
object


In [59]:
# make copy
fleet_raw = fleet.copy()
fleet_raw.shape

(63896, 23)

In [60]:
# rename columns
MEASURES = {"DWT_k": "Dead weight tons in thousands","Share_world": "Percentage of total world",
            "Share_fleet": "Percentage of total fleet",
            "Ships": "Number of ships",
            "GT_k": "Gross Tonnage in thousands",
            "Age": "Average age of vessels (years)"}

rename = {"Economy": "Code", "Economy Label": "Economy","ShipType": "ShipTypeCode", "ShipType Label": "ShipType"}
for short, long in MEASURES.items():
    rename[long] = short
    rename[long + " Footnote"] = short + "_footnote"
    rename[long + " Missing value"] = short + "_missing"

fleet = fleet_raw.rename(columns=rename)
print(fleet.columns.tolist())
fleet.head(3)

['Year', 'Code', 'Economy', 'ShipTypeCode', 'ShipType', 'DWT_k', 'DWT_k_footnote', 'DWT_k_missing', 'Share_world', 'Share_world_footnote', 'Share_world_missing', 'Share_fleet', 'Share_fleet_footnote', 'Share_fleet_missing', 'Ships', 'Ships_footnote', 'Ships_missing', 'GT_k', 'GT_k_footnote', 'GT_k_missing', 'Age', 'Age_footnote', 'Age_missing']


,Year,Code,Economy,ShipTypeCode,ShipType,DWT_k,DWT_k_footnote,DWT_k_missing,Share_world,Share_world_footnote,...,Share_fleet_missing,Ships,Ships_footnote,Ships_missing,GT_k,GT_k_footnote,GT_k_missing,Age,Age_footnote,Age_missing
0,1980,0000,World,0,Total fleet,672192.465,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
1,1980,0000,World,1,Oil tankers,337945.534,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected
2,1980,0000,World,2,Bulk carriers,181880.282,NaN,NaN,100.0,NaN,...,NaN,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected,NaN,NaN,No value reported or collected


In [61]:
# Extract the Break flags, one per measure plus a general one
for m in MEASURES:
    fleet["break_" + m] = fleet[m + "_footnote"].eq("Break")

fleet["series_break"] = fleet[["break_" + m for m in MEASURES]].any(axis=1)

print(fleet[["break_" + m for m in MEASURES]].sum())
print("any break:", fleet.series_break.sum())
print(fleet[fleet.series_break].groupby("Economy").Year.unique())

break_DWT_k          24
break_Share_world    24
break_Share_fleet    24
break_Ships          12
break_GT_k           12
break_Age            12
dtype: int64
any break: 24
Economy
France                  [2011]
United Kingdom          [1995]
United States     [2002, 2011]
Name: Year, dtype: object


In [62]:
# Keep the three missing-value reasons
for m in MEASURES:
    v, r = fleet[m], fleet[m + "_missing"]
    print(f"{m:12} value present with a reason: {(v.notna() & r.notna()).sum()} | "
          f"value empty with no reason: {(v.isna() & r.isna()).sum()} | "
          f"reasons: {r.value_counts().to_dict()}")

na = fleet[fleet.DWT_k_missing.eq("Not applicable")]
print(na.groupby("Economy").Year.agg(["min", "max", "size"]).head(12))

DWT_k        value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Share_world  value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Share_fleet  value present with a reason: 0 | value empty with no reason: 0 | reasons: {'Not applicable': 5082}
Ships        value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 34214, 'Not applicable': 5082}
GT_k         value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 34214, 'Not applicable': 5082}
Age          value present with a reason: 0 | value empty with no reason: 0 | reasons: {'No value reported or collected': 38791, 'Not applicable': 5082}
                                 min   max  size
Economy                                         
Azerbaijan                      1980  1991    72
Belarus                         1980  1991    72
Bosnia an

In [63]:
# Check that the numeric columns are numeric
print(fleet[list(MEASURES)].dtypes)

DWT_k          float64
Share_world    float64
Share_fleet    float64
Ships          float64
GT_k           float64
Age            float64
dtype: object


In [64]:
# Classify every row
def level(code):
    try:
        c = int(code)
    except ValueError:
        return "unallocated"            # '0000u' = World n.e.s.
    if c == 0:
        return "world"
    if c == 1 or c >= 1000:
        return "grouping"
    return "country"

fleet["level"] = fleet["Code"].map(level)
print(fleet["level"].value_counts())

level
country        44809
grouping       18683
world            282
unallocated      122
Name: count, dtype: int64


In [65]:
# ISO3 for country rows (numeric code first, name as a fallback)
import pycountry
ALIAS = {"Türkiye": "TUR", "Turkiye": "TUR", "Republic of Korea": "KOR", "Dem. People's Rep. of Korea": "PRK",
         "Dem. Rep. of the Congo": "COD", "Iran (Islamic Republic of)": "IRN", "Bolivia (Plurinational State of)": "BOL",
         "Lao People's Dem. Rep.": "LAO", "Russian Federation": "RUS", "Viet Nam": "VNM", "China, Hong Kong SAR": "HKG",
         "China, Taiwan Province of": "TWN", "China, Macao SAR": "MAC", "Cote d'Ivoire": "CIV"}

def iso3(code, name):
    hit = pycountry.countries.get(numeric=str(int(code)).zfill(3))
    if hit:
        return hit.alpha_3
    if name in ALIAS:
        return ALIAS[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

fleet["ISO3"] = [iso3(c, n) if l == "country" else None
                 for c, n, l in zip(fleet["Code"], fleet["Economy"], fleet["level"])]
no_iso = fleet[(fleet.level == "country") & fleet.ISO3.isna()]
print("country rows with no ISO3:", len(no_iso))
print(sorted(no_iso.Economy.unique()))

country rows with no ISO3: 2924
['Czechoslovakia', 'Democratic Republic of Germany', 'Ethiopia (...1991)', 'Indonesia (...2002)', 'Netherlands Antilles', 'Panama, excluding Canal Zone', 'Serbia and Montenegro', 'Sudan (...2011)', 'Union of Soviet Socialist Republics', 'Yemen, Arab Republic', 'Yemen, Democratic', 'Yugoslavia, Soc. Fed. Rep. of']


In [66]:
# Impossible average ages: investigate, then handle
bad = fleet[(fleet.Age <= 0) | (fleet.Age > 60)]
print(len(bad), "rows with Age <= 0 or > 60")
print(bad.groupby("Economy").Age.agg(["size", "max"]).sort_values("size", ascending=False).head(10))

fleet["Age_raw"] = fleet["Age"]
fleet["age_invalid"] = (fleet.Age <= 0) | (fleet.Age > 60)
fleet.loc[fleet.age_invalid, "Age"] = np.nan         # the original stays in Age_raw
print("age_invalid rows:", int(fleet.age_invalid.sum()))

216 rows with Age <= 0 or > 60
                          size     max
Economy                               
Turks and Caicos Islands    37  109.92
Anguilla                    24   84.92
Guatemala                   16   80.34
British Virgin Islands      16   81.25
Nicaragua                   15   79.92
Haiti                       15   72.00
Trinidad and Tobago         12   83.09
Grenada                     11   78.42
South Africa                 9   68.92
Suriname                     9   68.45
age_invalid rows: 216


In [83]:
#  Duplicates and consistency, one measure at a time

print("duplicate rows:", fleet.duplicated(["Year", "Code", "ShipType"]).sum(), "(expect 0)")

rows, failures = [], []
total_fleet = fleet[fleet.ShipType == "Total fleet"]
parts_fleet = fleet[fleet.ShipType != "Total fleet"]

for m in ("DWT_k", "GT_k", "Ships"):
    total = total_fleet.set_index(["Year", "Code"])[m]
    parts = parts_fleet.groupby(["Year", "Code"])[m].agg(sum=lambda s: s.sum(min_count=1), count="count")

    x = pd.concat([total.rename("total"), parts], axis=1).dropna(subset=["total", "sum"])
    x = x[x.total > 0]
    x["gap_pct"] = (x["sum"] / x.total - 1) * 100
    bad = x[x.gap_pct.abs() >= 1].assign(measure=m)
    failures.append(bad.reset_index())

    c = total_fleet[total_fleet.level == "country"].groupby("Year")[m].sum(min_count=1)
    w = total_fleet[total_fleet.level == "world"].set_index("Year")[m]
    ratio = (c / w.where(w > 0)).dropna()

    rows.append({
        "measure": m,
        "checked": len(x),
        "all 5 types present": round((x["count"] == 5).mean(), 3),
        "types sum to total (within 1%)": round((x.gap_pct.abs() < 1).mean(), 4),
        "failures": len(bad),
        "countries / World, min": round(ratio.min(), 3),
        "countries / World, latest": round(ratio.iloc[-1], 3),
    })

print(pd.DataFrame(rows).set_index("measure").to_string())
failures = pd.concat(failures, ignore_index=True)
failures.merge(fleet[["Code", "Economy"]].drop_duplicates(), on="Code").round(2)

duplicate rows: 0 (expect 0)
         checked  all 5 types present  types sum to total (within 1%)  failures  countries / World, min  countries / World, latest
measure                                                                                                                           
DWT_k      11115                0.686                          0.9992         9                   0.991                      0.991
GT_k        4100                1.000                          1.0000         0                   0.992                      0.992
Ships       4100                1.000                          1.0000         0                   0.979                      0.980


,Year,Code,total,sum,count,gap_pct,measure,Economy
0,1990,660,4.00,3.00,1,-25.00,DWT_k,Anguilla
1,1990,230,94.00,93.00,2,-1.06,DWT_k,Ethiopia (...1991)
2,1990,520,45.00,47.00,3,4.44,DWT_k,Nauru
3,1990,768,65.00,66.00,3,1.54,DWT_k,Togo
4,1991,768,78.00,79.00,3,1.28,DWT_k,Togo
5,2006,5420,63203.05,65369.28,5,3.43,DWT_k,Northern Europe
6,2006,826,11662.39,13828.61,5,18.57,DWT_k,United Kingdom
7,2006,3916,165288.65,167454.88,5,1.31,DWT_k,G20 (Group of Twenty)
8,2006,3905,212284.27,214450.50,5,1.02,DWT_k,OECD (Organisation for Economic Cooperation an...


#### 9 DWT failures: 5 rounding in small fleets (1990–91), 4 from UK 2006, where the types exceed the reported total; kept as reported

In [82]:
# Average age: flag small fleets, keep the values
fleet["Age_raw"] = fleet["Age"]
fleet["age_odd"] = (fleet.Age <= 0) | (fleet.Age > 60)
fleet["age_small_fleet"] = fleet.Age.notna() & fleet.Ships.le(5)

odd = fleet[fleet.age_odd]
print("odd ages (<= 0 or > 60):", len(odd))
print("  of which fleets with 5 ships or fewer:", int(odd.Ships.le(5).sum()))
print(odd.groupby("level").size().to_dict())
print(odd.sort_values("Age", ascending=False)[["Year", "Economy", "ShipType", "Ships", "GT_k", "Age"]].head(5).to_string(index=False))

# Age is kept as reported; use age_small_fleet to exclude one-ship fleets from age comparisons
print("age rows flagged small fleet:", int(fleet.age_small_fleet.sum()), "of", int(fleet.Age.notna().sum()))

odd ages (<= 0 or > 60): 0
  of which fleets with 5 ships or fewer: 0
{}
Empty DataFrame
Columns: [Year, Economy, ShipType, Ships, GT_k, Age]
Index: []
age rows flagged small fleet: 3039 of 19807


In [80]:
# Flag latest year and create analytical subset

# Flag the latest available year
LATEST = int(fleet["Year"].max())

fleet["latest_year_flag"] = fleet["Year"].eq(LATEST)

print("Latest year:", LATEST)
print("Rows flagged:", int(fleet["latest_year_flag"].sum()))

# Create country-level dataset
fleet_country = fleet[(fleet["level"] == "country") &fleet["ISO3"].notna()].copy()

print("\nAnalytical subset:")
print("Shape:", fleet_country.shape)
print("Unique countries:", fleet_country["ISO3"].nunique())
print("Year range:", fleet_country["Year"].min(), "-", fleet_country["Year"].max())
print("Latest-year records:", int(fleet_country["latest_year_flag"].sum()))

# Check economy classification
print("\nEconomy levels:")
print(fleet_country["level"].value_counts())

Latest year: 2026
Rows flagged: 1680

Analytical subset:
Shape: (41885, 37)
Unique countries: 208
Year range: 1980 - 2026
Latest-year records: 1188

Economy levels:
level
country    41885
Name: count, dtype: int64


In [81]:
# Save cleaned datasets
from pathlib import Path

PROC = Path("Datasets/processed")
PROC.mkdir(parents=True, exist_ok=True)

fleet.to_csv(PROC / "fleet_clean.csv", index=False)
fleet_country.to_csv(PROC / "fleet_country.csv", index=False)

print("Saved files:")
print(PROC / "fleet_clean.csv")
print(PROC / "fleet_country.csv")

Saved files:
Datasets\processed\fleet_clean.csv
Datasets\processed\fleet_country.csv


In [73]:
# LSCI
lsci.isnull().sum()

Economy_Label                                  0
Quarter_Label                                  0
Index_average_Q1_2023__100_Value             289
Index_average_Q1_2023__100_Footnote        15013
Index_average_Q1_2023__100_MissingValue    14724
dtype: int64

In [139]:
print("Dataset shape:", lsci.shape)

print("\nMissing-value reasons:")
print(lsci["Index_average_Q1_2023__100_MissingValue"].value_counts())

print("\nFootnotes:")
print(lsci["Index_average_Q1_2023__100_Footnote"].value_counts())

Dataset shape: (15013, 5)

Missing-value reasons:
Index_average_Q1_2023__100_MissingValue
Not applicable    289
Name: count, dtype: int64

Footnotes:
Series([], Name: count, dtype: int64)


In [140]:
# check duplicate economy-quarter records
print("Duplicate economy-quarter records:",lsci.duplicated(subset=["Economy_Label", "Quarter_Label"]).sum())

Duplicate economy-quarter records: 0


In [141]:
lsci.dtypes

Economy_Label                               object
Quarter_Label                               object
Index_average_Q1_2023__100_Value           float64
Index_average_Q1_2023__100_Footnote        float64
Index_average_Q1_2023__100_MissingValue     object
dtype: object

In [142]:
lsci_clean = lsci.drop(columns=["Index_average_Q1_2023__100_Footnote"]).rename(columns={
    "Economy_Label": "Economy",
    "Quarter_Label": "Quarter",
    "Index_average_Q1_2023__100_Value": "LSCI",
    "Index_average_Q1_2023__100_MissingValue": "LSCI_missing"
}).copy()

print("Cleaned shape:", lsci_clean.shape)
print(lsci_clean.columns.tolist())

Cleaned shape: (15013, 4)
['Economy', 'Quarter', 'LSCI', 'LSCI_missing']


In [143]:
# quarterly coverage
print("Unique economies:", lsci_clean["Economy"].nunique())
print("Unique quarters:", lsci_clean["Quarter"].nunique())

print("\nQuarterly coverage:")
print(lsci_clean["Quarter"].value_counts().sort_index().to_string())

print("\nQuarter range:")
print(lsci_clean["Quarter"].min(), "-", lsci_clean["Quarter"].max())

Unique economies: 191
Unique quarters: 83

Quarterly coverage:
Quarter
Q1 2006    179
Q1 2007    178
Q1 2008    177
Q1 2009    179
Q1 2010    179
Q1 2011    181
Q1 2012    179
Q1 2013    181
Q1 2014    180
Q1 2015    182
Q1 2016    180
Q1 2017    182
Q1 2018    182
Q1 2019    183
Q1 2020    183
Q1 2021    182
Q1 2022    182
Q1 2023    179
Q1 2024    183
Q1 2025    182
Q1 2026    181
Q2 2006    179
Q2 2007    180
Q2 2008    179
Q2 2009    180
Q2 2010    179
Q2 2011    180
Q2 2012    181
Q2 2013    181
Q2 2014    180
Q2 2015    181
Q2 2016    181
Q2 2017    182
Q2 2018    182
Q2 2019    183
Q2 2020    183
Q2 2021    182
Q2 2022    181
Q2 2023    180
Q2 2024    183
Q2 2025    182
Q2 2026    181
Q3 2006    178
Q3 2007    180
Q3 2008    180
Q3 2009    179
Q3 2010    181
Q3 2011    179
Q3 2012    181
Q3 2013    182
Q3 2014    183
Q3 2015    180
Q3 2016    182
Q3 2017    182
Q3 2018    182
Q3 2019    183
Q3 2020    182
Q3 2021    181
Q3 2022    181
Q3 2023    181
Q3 2024    181
Q3 2025    182

In [144]:
# extract year and quarter
lsci_clean["Year"] = (lsci_clean["Quarter"].str.extract(r"(\d{4})").astype(int))

lsci_clean["Quarter_Number"] = (lsci_clean["Quarter"].str.extract(r"Q([1-4])").astype(int))


print("Year range:", lsci_clean["Year"].min(), "-", lsci_clean["Year"].max())
print("Quarter range:", lsci_clean["Quarter_Number"].min(), "-", lsci_clean["Quarter_Number"].max())

Year range: 2006 - 2026
Quarter range: 1 - 4


In [145]:
print("Missing LSCI without reason:",(lsci_clean["LSCI"].isna() &lsci_clean["LSCI_missing"].isna()).sum())

print("Reason present but LSCI available:",(lsci_clean["LSCI"].notna() &lsci_clean["LSCI_missing"].notna()).sum())

Missing LSCI without reason: 0
Reason present but LSCI available: 0


In [146]:
lsci_clean["LSCI"].describe()

count    14724.000000
mean        96.932643
std        133.808579
min          2.500000
25%         22.792500
50%         47.060000
75%        106.880000
max       1341.970000
Name: LSCI, dtype: float64

In [147]:
print("Zero or negative LSCI values:",(lsci_clean["LSCI"] <= 0).sum())

Zero or negative LSCI values: 0


In [148]:
# inspect the highest values
lsci_clean.nlargest(10, "LSCI")[["Economy", "Quarter", "LSCI"]]

,Economy,Quarter,LSCI
2358,China,Q3 2026,1341.97
2356,China,Q1 2026,1322.63
2357,China,Q2 2026,1319.46
2355,China,Q4 2025,1312.61
2354,China,Q3 2025,1300.39
2353,China,Q2 2025,1291.28
2352,China,Q1 2025,1273.07
2351,China,Q4 2024,1250.45
2350,China,Q3 2024,1224.83
2349,China,Q2 2024,1195.99


In [149]:
print("Blank economy labels:",lsci_clean["Economy"].str.strip().eq("").sum())

print("Blank quarter labels:",lsci_clean["Quarter"].str.strip().eq("").sum())

Blank economy labels: 0
Blank quarter labels: 0


In [150]:
# save cleaned lsci
from pathlib import Path

Path("Datasets/processed").mkdir(parents=True, exist_ok=True)

lsci_clean.to_csv("Datasets/processed/LSCI_quarterly_cleaned.csv",index=False)

In [74]:
# port throughout
teu.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         3011
TEU (Twenty foot Equivalent Unit) Missing value    2900
dtype: int64

In [124]:
# economy structure
print("Unique economies:", teu["Economy"].nunique())

print("\nFirst 40 economy labels:")
print(teu["Economy Label"].drop_duplicates().head(40).to_list())

Unique economies: 202

First 40 economy labels:
['World', 'Africa', 'Northern Africa', 'Sub-Saharan Africa', 'Eastern Africa', 'Middle Africa', 'Southern Africa', 'Western Africa', 'Americas', 'Northern America', 'Latin America and the Caribbean', 'Caribbean', 'Central America', 'South America', 'Asia', 'Eastern Asia', 'South-eastern Asia', 'Southern Asia', 'Western Asia', 'Europe', 'Eastern Europe', 'Northern Europe', 'Southern Europe', 'Western Europe', 'Oceania', 'Australia and New Zealand', 'Oceania excluding Australia and New Zealand', 'Albania', 'Algeria', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Aruba', 'Australia', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belgium', 'Belize']


In [125]:
# create separate country/economy-level dataset
teu_economy = teu[teu["Economy"] < 1000].copy()

print("Original dataset:", teu.shape)
print("Economy-level dataset:", teu_economy.shape)
print("Unique economies:", teu_economy["Economy"].nunique())

Original dataset: (3030, 6)
Economy-level dataset: (2055, 6)
Unique economies: 137


In [127]:
teu_economy[["Economy", "Economy Label"]].drop_duplicates().sort_values("Economy")

,Economy,Economy Label
0,0,World
27,8,Albania
28,12,Algeria
29,24,Angola
30,28,Antigua and Barbuda
...,...,...
158,840,United States
159,858,Uruguay
160,862,Venezuela (Bolivarian Rep. of)
135,882,Samoa


In [128]:
# separate world aggregate from individual economies
teu_country = teu_economy[teu_economy["Economy"] != 0].copy()

print("Country-level dataset:", teu_country.shape)
print("Unique economies:", teu_country["Economy"].nunique())

Country-level dataset: (2040, 6)
Unique economies: 136


In [129]:
teu_country.isnull().sum()

Year                                                  0
Economy                                               0
Economy Label                                         0
TEU (Twenty foot Equivalent Unit)                   130
TEU (Twenty foot Equivalent Unit) Footnote         2021
TEU (Twenty foot Equivalent Unit) Missing value    1910
dtype: int64

In [130]:
teu_country["TEU (Twenty foot Equivalent Unit) Missing value"].value_counts()

TEU (Twenty foot Equivalent Unit) Missing value
Not publishable    99
Not applicable     31
Name: count, dtype: int64

In [131]:
teu_country["TEU (Twenty foot Equivalent Unit) Footnote"].value_counts()

TEU (Twenty foot Equivalent Unit) Footnote
Estimated                                                    15
Revised more than 20% in relation to the previous update.     4
Name: count, dtype: int64

In [132]:
teu_country.duplicated(subset=["Year", "Economy"]).sum()

np.int64(0)

In [133]:
(teu_country["TEU (Twenty foot Equivalent Unit)"] <= 0).sum()

np.int64(0)

In [134]:
print("Year range:", teu_country["Year"].min(), "-", teu_country["Year"].max())
print("Unique years:", teu_country["Year"].nunique())
print(teu_country["Year"].value_counts().sort_index())

Year range: 2010 - 2024
Unique years: 15
Year
2010    136
2011    136
2012    136
2013    136
2014    136
2015    136
2016    136
2017    136
2018    136
2019    136
2020    136
2021    136
2022    136
2023    136
2024    136
Name: count, dtype: int64


In [135]:
# rename columns
teu_country = teu_country.rename(columns={
    "Economy": "Economy_Code",
    "Economy Label": "Economy",
    "TEU (Twenty foot Equivalent Unit)": "TEU",
    "TEU (Twenty foot Equivalent Unit) Footnote": "TEU_footnote",
    "TEU (Twenty foot Equivalent Unit) Missing value": "TEU_missing"
})
teu_country.columns.tolist()

['Year', 'Economy_Code', 'Economy', 'TEU', 'TEU_footnote', 'TEU_missing']

In [136]:
teu_country.dtypes

Year              int64
Economy_Code      int64
Economy          object
TEU             float64
TEU_footnote     object
TEU_missing      object
dtype: object

In [137]:
print("Duplicate rows:", teu_country.duplicated().sum())

Duplicate rows: 0


In [138]:
# save processed dataset
from pathlib import Path

Path("Datasets/processed").mkdir(parents=True, exist_ok=True)

teu_country.to_csv("Datasets/processed/TEU_country_cleaned.csv",index=False)

In [75]:
# cppi
cppi.isnull().sum()

Port                                2
UN/LOCODE                           7
Territory                           7
World Bank Region                   7
Region average CPPI                 7
World Bank Income Group             7
Income Group average CPPI           7
Maritime services region            7
Region's average cppi               7
CPPI 2020                         100
CPPI 2021                          74
CPPI 2022                          86
CPPI 2023                          35
CPPI 2024                          23
CPPI 2025                          26
Rank 2025                          26
Berth hours in % of port hours     26
Statistical Index                  26
Administrative Index               26
Number of calls in sample          26
dtype: int64

In [92]:
# CPPI cleaning
cppi_clean = cppi.copy()

print("Original shape:", cppi.shape)
print("Working copy shape:", cppi_clean.shape)

print("Fully duplicated rows:", cppi_clean.duplicated().sum())

Original shape: (426, 20)
Working copy shape: (426, 20)
Fully duplicated rows: 1


In [93]:
# drop fully duplicated 
cppi_clean = cp.drop_duplicates().copy()

print("Shape after removing duplicates:", cppi_clean.shape)
print("Remaining fully duplicated rows:", cppi_clean.duplicated().sum())

Shape after removing duplicates: (425, 20)
Remaining fully duplicated rows: 0


In [95]:
# remove non-port rows
cppi_clean = cppi_clean.loc[~no_info].copy()

print("Shape after removing non-port rows:", cppi_clean.shape)
print("Missing UN/LOCODE:", cppi_clean["UN/LOCODE"].isna().sum())
print("Missing Territory:", cppi_clean["Territory"].isna().sum())

Shape after removing non-port rows: (419, 20)
Missing UN/LOCODE: 0
Missing Territory: 0


In [97]:
# standardize column names
cppi_clean["Port"] = cppi_clean["Port"].astype("string").str.strip()

cppi_clean = cppi_clean.rename(columns={
    "UN/LOCODE": "LOCODE",
    "Territory": "Country",
    "World Bank Region": "WB_region",
    "World Bank Income Group": "Income_group",
    "Maritime services region": "Maritime_region"
})
cppi_clean.columns.tolist()

['Port',
 'LOCODE',
 'Country',
 'WB_region',
 'Region average CPPI',
 'Income_group',
 'Income Group average CPPI',
 'Maritime_region',
 "Region's average cppi",
 'CPPI 2020',
 'CPPI 2021',
 'CPPI 2022',
 'CPPI 2023',
 'CPPI 2024',
 'CPPI 2025',
 'Rank 2025',
 'Berth hours in % of port hours',
 'Statistical Index',
 'Administrative Index',
 'Number of calls in sample']

In [98]:
missing = cppi_clean.isna().sum()

print("Missing values by column:")
print(missing[missing > 0].sort_values(ascending=False))

print("\nTotal missing values:", cppi_clean.isna().sum().sum())

Missing values by column:
CPPI 2020                         93
CPPI 2022                         79
CPPI 2021                         67
CPPI 2023                         28
CPPI 2025                         19
Rank 2025                         19
Berth hours in % of port hours    19
Statistical Index                 19
Administrative Index              19
Number of calls in sample         19
CPPI 2024                         16
dtype: int64

Total missing values: 397


In [99]:
missing_2025 = cppi_clean[
    cppi_clean["CPPI 2025"].isna()
]

print("Ports missing CPPI 2025:", len(missing_2025))

print(
    missing_2025[
        ["Port", "LOCODE", "Country", "CPPI 2020",
         "CPPI 2021", "CPPI 2022", "CPPI 2023", "CPPI 2024"]
    ].to_string(index=False)
)

Ports missing CPPI 2025: 19
              Port LOCODE      Country  CPPI 2020  CPPI 2021  CPPI 2022  CPPI 2023  CPPI 2024
              Aden  YEADE        Yemen        NaN      -21.0      -13.0        4.0      -24.0
           Berbera  SOBBO      Somalia        2.0       12.0       13.0       31.0       -3.0
           Bintulu  MYBTU     Malaysia        NaN        NaN        NaN      -77.0      -60.0
          Castries  LCCAS  Saint Lucia        NaN        NaN        NaN        1.0        3.0
       Gioia Tauro  ITGIT        Italy       60.0       46.0        8.0       13.0       60.0
          Halmstad  SEHAD       Sweden        NaN        NaN        NaN        NaN        6.0
         Heraklion  GRHER       Greece        4.0        4.0        3.0        1.0       -5.0
        Kalundborg  DKKAL      Denmark        NaN        NaN        NaN        NaN       22.0
King Abdullah Port  SAKAC Saudi Arabia      155.0      152.0      105.0      102.0       58.0
     Krishnapatnam  INKRI       

In [100]:
score_cols = [f"CPPI {y}" for y in range(2020, 2026)]

cppi_clean["years_available"] = cppi_clean[score_cols].notna().sum(axis=1)

print("Distribution of available CPPI years:")
print(cppi_clean["years_available"].value_counts().sort_index())

print("\nPorts with no CPPI scores:")
print(
    cppi_clean.loc[
        cppi_clean["years_available"] == 0,
        ["Port", "LOCODE", "Country"]
    ].to_string(index=False)
)

Distribution of available CPPI years:
years_available
1     14
2     16
3     31
4     20
5     35
6    303
Name: count, dtype: int64

Ports with no CPPI scores:
Empty DataFrame
Columns: [Port, LOCODE, Country]
Index: []


In [101]:
# numeric data types
numeric_cols = [
    "Region average CPPI",
    "Income Group average CPPI",
    "Region's average cppi",
    *score_cols,
    "Rank 2025",
    "Berth hours in % of port hours",
    "Statistical Index",
    "Administrative Index",
    "Number of calls in sample"
]

cppi_clean[numeric_cols].dtypes

Region average CPPI               float64
Income Group average CPPI         float64
Region's average cppi             float64
CPPI 2020                         float64
CPPI 2021                         float64
CPPI 2022                         float64
CPPI 2023                         float64
CPPI 2024                         float64
CPPI 2025                         float64
Rank 2025                         float64
Berth hours in % of port hours    float64
Statistical Index                 float64
Administrative Index              float64
Number of calls in sample         float64
dtype: object

In [102]:
print(cppi_clean[numeric_cols].describe().T.to_string())

                                count        mean         std         min         25%         50%         75%          max
Region average CPPI             419.0   -1.341200   29.907600  -71.972350  -14.986492   -8.086595   30.729639    34.424249
Income Group average CPPI       419.0   -1.359779    6.021963  -25.469787    1.362073    1.362073    1.615451     1.615451
Region's average cppi           419.0   -0.973744   36.738435  -98.080938  -19.239499   -7.718713   17.910438   103.168045
CPPI 2020                       326.0   37.710675   48.038120 -108.000000    6.000000   24.000000   68.000000   170.000000
CPPI 2021                       352.0    5.373509   80.082824 -665.000000   -7.000000   12.000000   39.000000   152.000000
CPPI 2022                       340.0   -2.009647   78.582603 -675.000000  -16.000000    7.860000   34.000000   139.000000
CPPI 2023                       391.0    6.866905   68.823008 -519.000000  -12.500000    6.000000   39.000000   152.000000
CPPI 2024       

In [103]:
# extreme observations
for col in score_cols:
    print(f"\n{col}")

    print("Lowest:")
    print(
        cppi_clean.nsmallest(3, col)[
            ["Port", "LOCODE", col]
        ].to_string(index=False)
    )

    print("Highest:")
    print(
        cppi_clean.nlargest(3, col)[
            ["Port", "LOCODE", col]
        ].to_string(index=False)
    )


CPPI 2020
Lowest:
          Port LOCODE  CPPI 2020
        Durban  ZADUR     -108.0
Port Elizabeth  ZAPLZ     -103.0
     Cape Town  ZACPT      -96.0
Highest:
              Port LOCODE  CPPI 2020
          Yokohama  JPYOK      170.0
King Abdullah Port  SAKAC      155.0
           Qingdao  CNQIN      148.0

CPPI 2021
Lowest:
       Port LOCODE  CPPI 2021
 Long Beach  USLGB     -665.0
Los Angeles  USLAX     -598.0
  Vancouver  CAVAN     -394.0
Highest:
              Port LOCODE  CPPI 2021
King Abdullah Port  SAKAC      152.0
           Salalah  OMSLL      143.0
        Hamad Port  QAHMD      138.0

CPPI 2022
Lowest:
      Port LOCODE  CPPI 2022
  Savannah  USSAV     -675.0
 Vancouver  CAVAN     -395.0
Long Beach  USLGB     -363.0
Highest:
        Port LOCODE  CPPI 2022
   Yangshan*  CNYSN      139.0
     Salalah  OMSLL      136.0
Khalifa Port  AEKHL      128.0

CPPI 2023
Lowest:
               Port LOCODE  CPPI 2023
          Cape Town  ZACPT     -519.0
Coega (Ngqura) Port  ZAZBA     -4

In [76]:
# us port calls arrivals
calls.isnull().sum()

Period                                   0
Period Label                             0
Economy                                  0
Economy Label                            0
CommercialMarket                         0
CommercialMarket Label                   0
Number of port calls                   490
Number of port calls Footnote         9300
Number of port calls Missing value    8810
dtype: int64

In [104]:
# 2025 supporting indicators
components = [
    "CPPI 2025",
    "Statistical Index",
    "Administrative Index",
    "Berth hours in % of port hours"
]

print(cppi_clean[components].corr().round(3))

                                CPPI 2025  Statistical Index  \
CPPI 2025                           1.000              0.994   
Statistical Index                   0.994              1.000   
Administrative Index                0.999              0.989   
Berth hours in % of port hours      0.311              0.316   

                                Administrative Index  \
CPPI 2025                                      0.999   
Statistical Index                              0.989   
Administrative Index                           1.000   
Berth hours in % of port hours                 0.307   

                                Berth hours in % of port hours  
CPPI 2025                                                0.311  
Statistical Index                                        0.316  
Administrative Index                                     0.307  
Berth hours in % of port hours                           1.000  


In [105]:
print(
    cppi_clean[
        ["CPPI 2025", "Statistical Index", "Administrative Index"]
    ].nunique()
)

CPPI 2025               400
Statistical Index       388
Administrative Index    400
dtype: int64


In [106]:
cppi_clean["Statistical Index"].value_counts().head(15)

Statistical Index
-3.27     2
-13.59    2
 11.84    2
-10.11    2
-1.50     2
 1.74     2
 2.90     2
 2.35     2
 3.59     2
 69.35    2
 3.09     2
 1.23     2
 55.91    1
 3.19     1
-3.98     1
Name: count, dtype: int64

In [107]:
categorical_cols = [
    "Country",
    "WB_region",
    "Income_group",
    "Maritime_region"
]

for col in categorical_cols:
    print(f"\n{col}")
    print("Unique values:", cppi_clean[col].nunique())
    print("Missing values:", cppi_clean[col].isna().sum())
    print(cppi_clean[col].value_counts(dropna=False).head(10))


Country
Unique values: 134
Missing values: 0
Country
United States     22
China             21
Brazil            15
Japan             15
Italy             15
Spain             14
Türkiye           13
India             12
United Kingdom     9
Chile              9
Name: count, dtype: int64

WB_region
Unique values: 7
Missing values: 0
WB_region
ECA       112
EAP       106
LAC        76
SSA        42
MENAAP     41
NAM        27
SAR        15
Name: count, dtype: int64

Income_group
Unique values: 5
Missing values: 0
Income_group
HI              209
UMI             120
LMI              74
LI               14
Unclassified      2
Name: count, dtype: int64

Maritime_region
Unique values: 20
Missing values: 0
Maritime_region
Mediterranean                  73
North Europe                   52
South East Asia                40
Caribbean & Central America    36
West Africa                    26
Australasia & Oceania          21
North America East Coast       21
South America East Coast       19
N

In [109]:
constant_cols = [
    col for col in cppi_clean.columns
    if cppi_clean[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_cols)

Constant columns: []


In [117]:
cppi_clean.columns.tolist()

['Port',
 'LOCODE',
 'Country',
 'WB_region',
 'Region average CPPI',
 'Income_group',
 'Income Group average CPPI',
 'Maritime_region',
 "Region's average cppi",
 'CPPI 2020',
 'CPPI 2021',
 'CPPI 2022',
 'CPPI 2023',
 'CPPI 2024',
 'CPPI 2025',
 'Rank 2025',
 'Berth hours in % of port hours',
 'Statistical Index',
 'Administrative Index',
 'Number of calls in sample',
 'years_available']

In [111]:
benchmark_cols = [
    "Region average CPPI",
    "Income Group average CPPI",
    "Region's average cppi"
]

print(cppi_clean[benchmark_cols].corr().round(3))

                           Region average CPPI  Income Group average CPPI  \
Region average CPPI                      1.000                      0.250   
Income Group average CPPI                0.250                      1.000   
Region's average cppi                    0.815                      0.259   

                           Region's average cppi  
Region average CPPI                        0.815  
Income Group average CPPI                  0.259  
Region's average cppi                      1.000  


In [116]:
print(cppi_clean[benchmark_cols].head(15).to_string(index=False))

 Region average CPPI  Income Group average CPPI  Region's average cppi
           -8.086595                   1.362073              -7.718713
          -71.972350                  -8.954771             -66.152693
          -14.986492                   1.615451             -19.239499
           30.729639                   1.362073             -11.011090
           18.958816                 -23.905865              63.164190
           18.958816                  -8.954771             -10.407568
           18.958816                   1.362073              42.058936
           18.958816                  -8.954771             -10.407568
           -8.086595                   1.362073             -10.407568
           18.958816                   1.615451             -10.407568
           -8.086595                   1.362073             -10.407568
          -14.986492                   1.615451              17.343876
           -8.086595                   1.615451             -10.407568
      

In [114]:
# correct integer-like columns
integer_cols = [
    "Rank 2025",
    "Number of calls in sample",
    "years_available"
]

cppi_clean[integer_cols] = cppi_clean[integer_cols].astype("Int64")

cppi_clean[integer_cols].dtypes

Rank 2025                    Int64
Number of calls in sample    Int64
years_available              Int64
dtype: object

In [118]:
print("Final shape:", cppi_clean.shape)

print("\nDuplicate rows:", cppi_clean.duplicated().sum())

print("\nMissing values:")
print(cppi_clean.isna().sum()[cppi_clean.isna().sum() > 0])

print("\nData types:")
print(cppi_clean.dtypes)

Final shape: (419, 21)

Duplicate rows: 0

Missing values:
CPPI 2020                         93
CPPI 2021                         67
CPPI 2022                         79
CPPI 2023                         28
CPPI 2024                         16
CPPI 2025                         19
Rank 2025                         19
Berth hours in % of port hours    19
Statistical Index                 19
Administrative Index              19
Number of calls in sample         19
dtype: int64

Data types:
Port                              string[python]
LOCODE                                    object
Country                                   object
WB_region                                 object
Region average CPPI                      float64
Income_group                              object
Income Group average CPPI                float64
Maritime_region                           object
Region's average cppi                    float64
CPPI 2020                                float64
CPPI 2021         

In [122]:
from pathlib import Path

Path("Datasets/processed").mkdir(parents=True, exist_ok=True)

cppi_clean.to_csv("Datasets/processed/CPPI_cleaned.csv", index=False)

In [151]:
# port calls
calls.isnull().sum()

Period                                   0
Period Label                             0
Economy                                  0
Economy Label                            0
CommercialMarket                         0
CommercialMarket Label                   0
Number of port calls                   490
Number of port calls Footnote         9300
Number of port calls Missing value    8810
dtype: int64

In [152]:
print("Original shape:", calls.shape)
print("\nColumns:")
print(calls.columns.tolist())

print("\nData types:")
print(calls.dtypes)

print("\nMissing values:")
print(calls.isnull().sum())

Original shape: (9300, 9)

Columns:
['Period', 'Period Label', 'Economy', 'Economy Label', 'CommercialMarket', 'CommercialMarket Label', 'Number of port calls', 'Number of port calls Footnote', 'Number of port calls Missing value']

Data types:
Period                                 object
Period Label                           object
Economy                                 int64
Economy Label                          object
CommercialMarket                        int64
CommercialMarket Label                 object
Number of port calls                  float64
Number of port calls Footnote         float64
Number of port calls Missing value     object
dtype: object

Missing values:
Period                                   0
Period Label                             0
Economy                                  0
Economy Label                            0
CommercialMarket                         0
CommercialMarket Label                   0
Number of port calls                   490
Number of

In [153]:
# Investigate the 490 missing port-call values
calls["Number of port calls Missing value"].value_counts()

Number of port calls Missing value
Not available or not separately reported    310
Not applicable                              108
Not publishable                              72
Name: count, dtype: int64

In [154]:
# check duplicates
print("Duplicate records:",calls.duplicated(subset=["Period", "Economy", "CommercialMarket"]).sum())

Duplicate records: 0


In [155]:
# check invalid port-call values
print("Zero or negative port calls:",(calls["Number of port calls"] <= 0).sum())

Zero or negative port calls: 0


In [156]:
# create the cleaned working copy
calls_clean = calls.drop(columns=["Number of port calls Footnote"]).rename(columns={"Number of port calls": "Port_Calls","Number of port calls Missing value": "Port_Calls_missing"}).copy()

print("Cleaned shape:", calls_clean.shape)
print(calls_clean.columns.tolist())

Cleaned shape: (9300, 8)
['Period', 'Period Label', 'Economy', 'Economy Label', 'CommercialMarket', 'CommercialMarket Label', 'Port_Calls', 'Port_Calls_missing']


In [157]:
# check period and category coverage
print("Unique periods:", calls_clean["Period"].nunique())
print("Period range:", calls_clean["Period Label"].min(),
      "-", calls_clean["Period Label"].max())

print("Unique economies:", calls_clean["Economy"].nunique())
print("Unique commercial markets:", calls_clean["CommercialMarket"].nunique())

print("\nPeriod coverage:")
print(calls_clean["Period Label"].value_counts().sort_index().to_string())

Unique periods: 12
Period range: S1   2018 - S2   2023
Unique economies: 87
Unique commercial markets: 9

Period coverage:
Period Label
S1   2018    773
S1   2019    773
S1   2020    774
S1   2021    774
S1   2022    777
S1   2023    777
S2   2018    774
S2   2019    774
S2   2020    775
S2   2021    776
S2   2022    776
S2   2023    777


In [158]:
# inspect the commercial market categories
calls_clean[["CommercialMarket", "CommercialMarket Label"]].drop_duplicates().sort_values("CommercialMarket")

,CommercialMarket,CommercialMarket Label
0,0,All ships
8,1,Passenger ships
1,2,Liquid bulk carriers
7,3,Container ships
5,4,Dry breakbulk carriers
4,5,Dry bulk carriers
6,6,Roll-on/ roll-off ships
2,7,Liquefied petroleum gas carriers
3,8,Liquefied natural gas carriers


In [159]:
# verify identifier consistency
print("Economy codes with multiple names:",(calls_clean.groupby("Economy")["Economy Label"].nunique() > 1).sum())

print("Market codes with multiple names:",(calls_clean.groupby("CommercialMarket")["CommercialMarket Label"].nunique() > 1).sum())

Economy codes with multiple names: 0
Market codes with multiple names: 0


In [160]:
# verify period identifiers
print("Period codes with multiple labels:",(calls_clean.groupby("Period")["Period Label"].nunique() > 1).sum())

print("Period labels with multiple codes:",(calls_clean.groupby("Period Label")["Period"].nunique() > 1).sum())

Period codes with multiple labels: 0
Period labels with multiple codes: 0


In [167]:
code_positions = [i for i, col in enumerate(calls_clean.columns)if col == "Code"]

calls_clean.columns.values[code_positions[1]] = "Economy"

print(calls_clean.columns.tolist())

['Period', 'Period Label', 'Code', 'Economy', 'MarketCode', 'Market', 'Port_calls', 'Port_calls_missing', 'Year', 'Half']


In [171]:
# Correct the second Code column to Economy
cols = calls_clean.columns.tolist()

code_positions = [i for i, col in enumerate(cols) if col == "Code"]

cols[code_positions[1]] = "Economy"
calls_clean.columns = cols

print("Columns:", calls_clean.columns.tolist())
print("Duplicate columns:", calls_clean.columns[calls_clean.columns.duplicated()].tolist())

Columns: ['Period', 'Period Label', 'Code', 'Economy', 'MarketCode', 'Market', 'Port_calls', 'Port_calls_missing', 'Year', 'Half']
Duplicate columns: []


In [168]:
def level_from_code(code):
    try:
        c = int(code)
    except (ValueError, TypeError):
        return "unallocated"

    return "world" if c == 0 else (
        "grouping" if c == 1 or c >= 1000 else "country"
    )

In [173]:
# Repair duplicate Code columns, if present
cols = calls_clean.columns.tolist()
code_positions = [i for i, col in enumerate(cols) if col == "Code"]

if len(code_positions) == 2:
    cols[code_positions[1]] = "Economy"
    calls_clean.columns = cols

# Confirm column names are unique
print("Duplicate columns:",
      calls_clean.columns[calls_clean.columns.duplicated()].tolist())

# Classify economies
code_series = calls_clean.iloc[:, calls_clean.columns.get_loc("Code")]
calls_clean["level"] = code_series.map(level_from_code)

print("Final shape:", calls_clean.shape)
print("\nEconomy classification:")
print(calls_clean["level"].value_counts().to_dict())

Duplicate columns: []
Final shape: (9300, 11)

Economy classification:
{'grouping': 6912, 'country': 2280, 'world': 108}


In [176]:
calls_clean.groupby("Period")["Port_calls"].apply(lambda s: s.isna().mean()).round(2).to_string()

'Period\n2018S01    0.05\n2018S02    0.05\n2019S01    0.05\n2019S02    0.06\n2020S01    0.05\n2020S02    0.06\n2021S01    0.06\n2021S02    0.06\n2022S01    0.05\n2022S02    0.05\n2023S01    0.04\n2023S02    0.05'

In [177]:
save(calls_clean, "portcalls_clean")

'portcalls_clean)' was not found in history, as a file, url, nor in the user namespace.


In [178]:
from pathlib import Path

output_dir = Path("Datasets/processed")
output_dir.mkdir(parents=True, exist_ok=True)

file_path = output_dir / "portcalls_clean.csv"

calls_clean.to_csv(file_path, index=False)

print("Saved successfully:", file_path.resolve())
print("Dataset shape:", calls_clean.shape)

Saved successfully: C:\Users\user\Documents\Global-Maritime-Trade-Intelligence\Datasets\processed\portcalls_clean.csv
Dataset shape: (9300, 11)


In [179]:
# chokepoint data
choke.isnull().sum()

date                      0
year                      0
month                     0
day                       0
portid                    0
portname                  0
n_container               0
n_dry_bulk                0
n_general_cargo           0
n_roro                    0
n_tanker                  0
n_cargo                   0
n_total                   0
capacity_container        0
capacity_dry_bulk         0
capacity_general_cargo    0
capacity_roro             0
capacity_tanker           0
capacity_cargo            0
capacity                  0
ObjectId                  0
dtype: int64

In [180]:
# copy of daily chokepoints
ck = choke.copy()

In [181]:
# convert dates into proper datetime format
ck["date"] = pd.to_datetime(ck["date"], errors="coerce")

In [182]:
# identify partial year
ck["partial_year"] = ck["year"] == ck["date"].max().year

In [183]:
# remove redudant columns
ck = ck.drop(columns=["year", "month", "day", "ObjectId"])

In [184]:
# sort observations
ck = ck.sort_values(["portid", "date"]).reset_index(drop=True)

In [185]:
# save cleaned 
ck.to_csv("Datasets/processed/chokepoints_cleaned.csv",index=False)

## EDA

In [ ]:
# Seaborne


# Modelling 

# Evaluation

# Deployment